# 🎙️ UVR5 MDX23C — Vocal Separation Pipeline (Kaggle GPU)
### Thực chiến: Tách vocal chất lượng production cho "50 Năm Về Sau"
**Yêu cầu Kaggle:** Accelerator = GPU T4 x1 hoặc P100 | Runtime ≥ 9 giờ

---
**Pipeline tổng quan:**
```
YouTube URL
  → yt-dlp (tải WAV 16-bit 44.1kHz)
  → MDX23C-8KFFT-3_7C  (pass 1: Vocal / Instrumental split)
  → UVR-DeEcho-DeReverb (pass 2: Loại echo/reverb còn sót)
  → UVR-BVE-4B_SRVgg    (pass 3: Loại backing vocal nếu cần)
  → Post-process: DeNoise + Loudness Normalize (LUFS -16)
  → Export: vocal_clean.wav + instrumental.wav
```

In [ ]:
# ============================================================
# CELL 1: KIỂM TRA GPU & CUDA
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================
# CELL 2: CÀI ĐẶT TẤT CẢ DEPENDENCIES
# Chạy 1 lần — mất ~3–5 phút
# ============================================================
import os

# audio-separator: wrapper Python chính thức cho UVR5 models
# phiên bản 0.24+ hỗ trợ MDX23C natively trên GPU
!pip install -q "audio-separator[gpu]"==0.24.2

# yt-dlp để tải audio từ YouTube
!pip install -q yt-dlp

# ffmpeg backend (Kaggle đã có, nhưng cài lại để chắc)
!apt-get install -y -q ffmpeg

# soundfile + librosa cho post-processing
!pip install -q soundfile librosa

# pyloudnorm cho LUFS normalization
!pip install -q pyloudnorm

# noisereduce cho denoising nhẹ
!pip install -q noisereduce

print("\n✅ Cài xong tất cả dependencies")

In [ ]:
# ============================================================
# CELL 3: CẤU HÌNH ĐƯỜNG DẪN & BIẾN TOÀN CỤC
# ============================================================
import os
from pathlib import Path

# ─── Thay URL này bằng link YouTube của bài bạn cần xử lý ───
YOUTUBE_URL = "https://www.youtube.com/watch?v=NHẬP_VIDEO_ID_Ở_ĐÂY"
# Ví dụ thực tế:
# YOUTUBE_URL = "https://www.youtube.com/watch?v=abcXYZ123"

# ─── Tên project (dùng để đặt tên file) ───
PROJECT_NAME = "50_nam_ve_sau"

# ─── Thư mục làm việc ───
BASE_DIR   = Path("/kaggle/working")
AUDIO_DIR  = BASE_DIR / "audio"
OUTPUT_DIR = BASE_DIR / "output"
MODEL_DIR  = BASE_DIR / "models"

for d in [AUDIO_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── File paths ───
RAW_AUDIO       = AUDIO_DIR / f"{PROJECT_NAME}_raw.wav"
VOCAL_PASS1     = OUTPUT_DIR / f"{PROJECT_NAME}_vocal_p1.wav"
INST_PASS1      = OUTPUT_DIR / f"{PROJECT_NAME}_instrumental_p1.wav"
VOCAL_PASS2     = OUTPUT_DIR / f"{PROJECT_NAME}_vocal_p2_dereverb.wav"
VOCAL_FINAL     = OUTPUT_DIR / f"{PROJECT_NAME}_VOCAL_FINAL.wav"
INST_FINAL      = OUTPUT_DIR / f"{PROJECT_NAME}_INSTRUMENTAL_FINAL.wav"

print("✅ Cấu hình xong")
print(f"   Working dir : {BASE_DIR}")
print(f"   Output dir  : {OUTPUT_DIR}")

In [ ]:
# ============================================================
# CELL 4: TẢI AUDIO TỪ YOUTUBE
# Tải định dạng WAV / FLAC chất lượng cao nhất có thể
# ============================================================
import subprocess

ydl_cmd = [
    "yt-dlp",
    "--no-playlist",
    "-x",                          # extract audio only
    "--audio-format", "wav",        # output WAV
    "--audio-quality", "0",         # best quality
    "--postprocessor-args",
    # ffmpeg args: PCM 32-bit float, 44100Hz, stereo
    "ffmpeg:-ar 44100 -ac 2 -sample_fmt s32",
    "-o", str(RAW_AUDIO.with_suffix(".%(ext)s")),
    YOUTUBE_URL
]

print("⏳ Đang tải audio từ YouTube...")
result = subprocess.run(ydl_cmd, capture_output=True, text=True)

if result.returncode == 0:
    # Tìm file vừa tải (extension có thể khác)
    candidates = list(AUDIO_DIR.glob(f"{PROJECT_NAME}_raw.*"))
    if candidates:
        downloaded = candidates[0]
        print(f"✅ Tải xong: {downloaded}")
        print(f"   Kích thước: {downloaded.stat().st_size / 1e6:.1f} MB")
    else:
        print("⚠️ Không tìm thấy file tải về — kiểm tra URL")
else:
    print("❌ Lỗi tải:")
    print(result.stderr[-2000:])

In [ ]:
# ============================================================
# CELL 4B: (TÙY CHỌN) UPLOAD FILE LOCAL THAY VÌ YOUTUBE
# Bỏ qua nếu đã chạy Cell 4 thành công
# ============================================================
# Nếu bạn có sẵn file audio:
#   1. Upload lên Kaggle Dataset hoặc dùng /kaggle/input/
#   2. Chỉnh đường dẫn bên dưới

# LOCAL_FILE = Path("/kaggle/input/your-dataset/song.mp3")
# 
# import subprocess
# subprocess.run([
#     "ffmpeg", "-i", str(LOCAL_FILE),
#     "-ar", "44100", "-ac", "2", "-sample_fmt", "s32",
#     "-y", str(RAW_AUDIO)
# ], check=True)
# print(f"✅ Đã convert sang WAV: {RAW_AUDIO}")
pass

In [ ]:
# ============================================================
# CELL 5: KIỂM TRA FILE AUDIO GỐC
# ============================================================
import librosa
import soundfile as sf
import numpy as np

# Tìm file raw
raw_candidates = list(AUDIO_DIR.glob(f"{PROJECT_NAME}_raw.*"))
assert raw_candidates, "❌ Không có file audio! Chạy lại Cell 4."
RAW_AUDIO_ACTUAL = raw_candidates[0]

info = sf.info(str(RAW_AUDIO_ACTUAL))
print("📊 Thông tin audio gốc:")
print(f"   File     : {RAW_AUDIO_ACTUAL.name}")
print(f"   Duration : {info.duration:.1f}s ({info.duration/60:.1f} phút)")
print(f"   Samplerate: {info.samplerate} Hz")
print(f"   Channels : {info.channels}")
print(f"   Format   : {info.format} / {info.subtype}")
print(f"   Filesize : {RAW_AUDIO_ACTUAL.stat().st_size / 1e6:.1f} MB")

# Nếu sample rate ≠ 44100, resample
if info.samplerate != 44100:
    print(f"\n⚠️ Samplerate {info.samplerate}Hz → resampling sang 44100Hz...")
    y, sr = librosa.load(str(RAW_AUDIO_ACTUAL), sr=44100, mono=False)
    sf.write(str(RAW_AUDIO_ACTUAL), y.T, 44100, subtype='PCM_32')
    print("✅ Resample xong")

In [ ]:
# ============================================================
# CELL 6: PASS 1 — MDX23C-8KFFT-3_7C (VOCAL / INSTRUMENTAL)
# Model tốt nhất trong dòng MDX23C cho nhạc có vocal rõ
# Tự tải model ~400MB lần đầu, cache lại những lần sau
# ============================================================
from audio_separator.separator import Separator
import torch

print("🚀 PASS 1: MDX23C-8KFFT-3_7C — Tách Vocal / Instrumental")
print("   (Model sẽ tự tải nếu chưa có — ~400MB)\n")

sep_p1 = Separator(
    model_file_dir=str(MODEL_DIR),
    output_dir=str(OUTPUT_DIR),
    output_format="WAV",

    # ─── MDX23C params tối ưu chất lượng ───
    mdx23c_params={
        "segment_size"      : 256,   # Tăng từ 256 lên giúp bắt được context dài hơn
        "overlap"           : 8,     # Overlap cao = ít artifact ở biên chunk
        "batch_size"        : 2,     # T4 = 16GB VRAM → batch_size=2 an toàn
        "enable_denoise"    : True,  # Khử nhiễu nội bộ model
        "compensation"      : 1.035, # Giá trị chuẩn của MDX23C-3_7C
    },

    # ─── Audio quality ───
    sample_rate=44100,
    normalization_threshold=0.9,
    amplification_threshold=0.0,

    # ─── Device ───
    use_cuda=torch.cuda.is_available(),
    use_cpu=not torch.cuda.is_available(),
)

# Load model
sep_p1.load_model("MDX23C-8KFFT-3_7C.ckpt")

# Chạy tách
p1_outputs = sep_p1.separate(str(RAW_AUDIO_ACTUAL))

print("\n✅ Pass 1 xong. Output files:")
for f in p1_outputs:
    fpath = Path(f)
    print(f"   {fpath.name}  ({fpath.stat().st_size/1e6:.1f} MB)")

In [ ]:
# ============================================================
# CELL 7: XÁC ĐỊNH FILE VOCAL TỪ PASS 1
# audio-separator đặt tên theo format: [stem]_(Vocals)_MDX23C.wav
# ============================================================
from pathlib import Path

p1_files = list(OUTPUT_DIR.glob("*MDX23C*"))
print("Files từ Pass 1:")
for f in p1_files:
    print(f"  {f.name}")

# Tìm vocal và instrumental
vocal_p1 = None
inst_p1  = None

for f in p1_files:
    name_lower = f.name.lower()
    if "vocal" in name_lower and "instru" not in name_lower:
        vocal_p1 = f
    elif "instru" in name_lower or "no_vocal" in name_lower:
        inst_p1 = f

# Fallback: nếu tên file không chứa 'vocal'/'instru',
# audio-separator đặt (Vocals) và (Instrumental)
if not vocal_p1:
    for f in p1_files:
        if "(Vocals)" in f.name:
            vocal_p1 = f
        elif "(Instrumental)" in f.name:
            inst_p1 = f

assert vocal_p1, "❌ Không tìm được file Vocal từ Pass 1!"
assert inst_p1,  "❌ Không tìm được file Instrumental từ Pass 1!"

print(f"\n🎤 Vocal P1     : {vocal_p1.name}")
print(f"🎵 Instrumental  : {inst_p1.name}")

In [ ]:
# ============================================================
# CELL 8: PASS 2 — UVR-DeEcho-DeReverb
# Loại bỏ echo và reverb phòng thu còn sót trong vocal track
# Cực kỳ quan trọng nếu bài gốc có nhiều reverb/không gian
# ============================================================
from audio_separator.separator import Separator
import torch

print("🚀 PASS 2: UVR-DeEcho-DeReverb — Loại echo/reverb khỏi vocal")

sep_p2 = Separator(
    model_file_dir=str(MODEL_DIR),
    output_dir=str(OUTPUT_DIR),
    output_format="WAV",

    # VR Architecture params
    vr_params={
        "batch_size"      : 4,      # DeEcho model nhẹ hơn, batch lớn hơn được
        "window_size"     : 512,    # 512 = balance quality/speed
        "aggression"      : 10,     # 0-100: mức độ loại reverb (10 = nhẹ tay, giữ âm sắc)
        "enable_tta"      : True,   # Test-time augmentation = chất lượng cao hơn (+30% thời gian)
        "enable_post_process": True,# Post-processing nội bộ
        "post_process_threshold": 0.2,
        "high_end_process": True,   # Bảo vệ tần số cao (sibilance)
    },

    sample_rate=44100,
    use_cuda=torch.cuda.is_available(),
    use_cpu=not torch.cuda.is_available(),
)

sep_p2.load_model("UVR-DeEcho-DeReverb.pth")
p2_outputs = sep_p2.separate(str(vocal_p1))

print("\n✅ Pass 2 xong:")
for f in p2_outputs:
    fpath = Path(f)
    print(f"   {fpath.name}  ({fpath.stat().st_size/1e6:.1f} MB)")

# Xác định file vocal sau dereverb
p2_files   = [Path(f) for f in p2_outputs]
vocal_p2   = None
for f in p2_files:
    name_lower = f.name.lower()
    # File "no reverb" / "dry" / "vocals" là kết quả cần
    if "no_reverb" in name_lower or "dry" in name_lower:
        vocal_p2 = f
    elif "(Vocals)" in f.name and "reverb" not in name_lower:
        vocal_p2 = f

# Fallback: lấy file nhỏ hơn (reverb track thường lớn hơn)
if not vocal_p2 and p2_files:
    vocal_p2 = min(p2_files, key=lambda x: x.stat().st_size)

print(f"\n🎤 Vocal sau DeReverb: {vocal_p2.name}")

In [ ]:
# ============================================================
# CELL 9: PASS 3 — UVR-BVE-4B_SRVgg (Backing Vocal Extractor)
# (TÙY CHỌN) Chỉ chạy nếu vocal track còn lẫn harmony/backing
# Bỏ qua cell này nếu chỉ cần vocal lead chính
# ============================================================
from audio_separator.separator import Separator
import torch

REMOVE_BACKING_VOCALS = True  # ← Đặt False nếu muốn giữ lại harmony

if REMOVE_BACKING_VOCALS:
    print("🚀 PASS 3: UVR-BVE-4B_SRVgg — Tách Lead Vocal khỏi Backing Vocal")

    sep_p3 = Separator(
        model_file_dir=str(MODEL_DIR),
        output_dir=str(OUTPUT_DIR),
        output_format="WAV",

        vr_params={
            "batch_size"      : 4,
            "window_size"     : 512,
            "aggression"      : 10,
            "enable_tta"      : True,
            "high_end_process": True,
        },

        sample_rate=44100,
        use_cuda=torch.cuda.is_available(),
        use_cpu=not torch.cuda.is_available(),
    )

    sep_p3.load_model("UVR-BVE-4B_SRVgg-300317-224840.pth")
    p3_outputs = sep_p3.separate(str(vocal_p2))

    print("\n✅ Pass 3 xong:")
    p3_files = [Path(f) for f in p3_outputs]
    for f in p3_files:
        print(f"   {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

    # Lấy lead vocal (thường là file nhỏ hơn)
    vocal_lead = None
    for f in p3_files:
        if "lead" in f.name.lower() or "primary" in f.name.lower():
            vocal_lead = f
    if not vocal_lead:
        vocal_lead = min(p3_files, key=lambda x: x.stat().st_size)

    print(f"\n🎤 Lead Vocal: {vocal_lead.name}")
    vocal_for_postprocess = vocal_lead
else:
    print("⏭️ Bỏ qua Pass 3 (REMOVE_BACKING_VOCALS = False)")
    vocal_for_postprocess = vocal_p2

print(f"\n→ Vocal sẽ đưa vào post-processing: {vocal_for_postprocess.name}")

In [ ]:
# ============================================================
# CELL 10: POST-PROCESSING — DENOISE + EQ + NORMALIZE
# Mục tiêu: Vocal sạch, không artifact, level chuẩn -16 LUFS
# ============================================================
import numpy as np
import soundfile as sf
import librosa
import noisereduce as nr
import pyloudnorm as pyln

print("🔧 Post-processing: DeNoise + EQ + Normalize...")

# ─── Load vocal ───
y, sr = librosa.load(str(vocal_for_postprocess), sr=44100, mono=False)

# audio-separator output: shape (samples,) nếu mono
# hoặc (2, samples) nếu stereo — chuẩn hóa shape
if y.ndim == 1:
    y = np.stack([y, y])  # mono → stereo

print(f"   Loaded: shape={y.shape}, sr={sr}Hz")

# ─── BƯỚC 1: Spectral Gating DeNoise ───
# Lấy 0.5s đầu làm noise profile (thường là khoảng lặng hoặc breath)
# prop_decrease=0.6: giảm 60% noise → không bị "pumping" artifact
print("   → DeNoise (spectral gating)...")
y_denoised = np.zeros_like(y)
for ch in range(y.shape[0]):
    noise_sample = y[ch, :int(sr * 0.5)]
    y_denoised[ch] = nr.reduce_noise(
        y=y[ch],
        y_noise=noise_sample,
        sr=sr,
        prop_decrease=0.55,    # Nhẹ tay — giữ âm sắc tự nhiên
        stationary=False,      # Non-stationary: phát hiện noise thay đổi theo thời gian
        freq_mask_smooth_hz=500,
        time_mask_smooth_ms=50,
        n_std_thresh_stationary=1.5,
    )

# ─── BƯỚC 2: High-pass filter (cắt rumble dưới 80Hz) ───
print("   → High-pass filter @ 80Hz...")
from scipy.signal import butter, sosfilt

def highpass_filter(y, sr, cutoff=80, order=4):
    sos = butter(order, cutoff / (sr / 2), btype='high', output='sos')
    return sosfilt(sos, y)

y_filtered = np.zeros_like(y_denoised)
for ch in range(y_denoised.shape[0]):
    y_filtered[ch] = highpass_filter(y_denoised[ch], sr, cutoff=80)

# ─── BƯỚC 3: De-esser nhẹ (giảm sibilance 6kHz-10kHz) ───
print("   → De-esser nhẹ (6kHz–10kHz)...")
from scipy.signal import butter, sosfilt

def deesser_band(y, sr, low_freq=5500, high_freq=10000, reduction_db=3.0):
    """Spectral de-esser đơn giản: giảm band 5.5k–10kHz theo ngưỡng"""
    reduction_linear = 10 ** (-reduction_db / 20)
    # Bandpass để detect sibilance
    sos_bp = butter(4, [low_freq/(sr/2), high_freq/(sr/2)], btype='band', output='sos')
    # Lowpass để lấy phần còn lại
    sos_lp = butter(4, low_freq/(sr/2), btype='low', output='sos')
    sos_hp = butter(4, high_freq/(sr/2), btype='high', output='sos')

    sibilant = sosfilt(sos_bp, y)
    low_end  = sosfilt(sos_lp, y)
    high_end = sosfilt(sos_hp, y)

    # Giảm sibilance
    sibilant_reduced = sibilant * reduction_linear
    return low_end + sibilant_reduced + high_end

y_deessed = np.zeros_like(y_filtered)
for ch in range(y_filtered.shape[0]):
    y_deessed[ch] = deesser_band(y_filtered[ch], sr,
                                   low_freq=5500, high_freq=10000,
                                   reduction_db=2.5)

# ─── BƯỚC 4: LUFS Normalization (-16 LUFS — chuẩn YouTube) ───
print("   → LUFS Normalization → -16 LUFS (chuẩn YouTube/Spotify)...")
meter = pyln.Meter(sr)  # ITU-R BS.1770-4

# pyloudnorm cần shape (samples, channels)
y_for_meter = y_deessed.T  # (samples, 2)

try:
    loudness = meter.integrated_loudness(y_for_meter)
    print(f"      Loudness hiện tại: {loudness:.1f} LUFS")

    TARGET_LUFS = -16.0
    y_normalized_T = pyln.normalize.loudness(y_for_meter, loudness, TARGET_LUFS)
    y_final = y_normalized_T.T  # back to (2, samples)

    # Peak limiter: không để vượt -1 dBFS
    peak = np.max(np.abs(y_final))
    if peak > 0.891:  # 0.891 ≈ -1dBFS
        y_final = y_final * (0.891 / peak)
        print(f"      Peak limiting applied (peak was {20*np.log10(peak):.1f} dBFS)")

    loudness_final = meter.integrated_loudness(y_final.T)
    print(f"      Loudness sau normalize: {loudness_final:.1f} LUFS")
except Exception as e:
    print(f"      ⚠️ LUFS normalize lỗi: {e} — dùng peak normalize")
    peak = np.max(np.abs(y_deessed))
    y_final = y_deessed * (0.891 / peak) if peak > 0 else y_deessed

# ─── Lưu vocal final ───
sf.write(str(VOCAL_FINAL), y_final.T, sr, subtype='PCM_24')
print(f"\n✅ Vocal FINAL đã lưu: {VOCAL_FINAL.name}")
print(f"   Size: {VOCAL_FINAL.stat().st_size/1e6:.1f} MB")

In [ ]:
# ============================================================
# CELL 11: XỬ LÝ INSTRUMENTAL FINAL
# Normalize -16 LUFS cho bản nhạc nền (dùng ghép video)
# ============================================================
import numpy as np
import soundfile as sf
import librosa
import pyloudnorm as pyln

print("🎵 Normalize Instrumental...")

yi, sri = librosa.load(str(inst_p1), sr=44100, mono=False)
if yi.ndim == 1:
    yi = np.stack([yi, yi])

meter = pyln.Meter(sri)
loudness_i = meter.integrated_loudness(yi.T)
print(f"   Loudness gốc: {loudness_i:.1f} LUFS")

yi_norm = pyln.normalize.loudness(yi.T, loudness_i, -16.0)
peak_i = np.max(np.abs(yi_norm))
if peak_i > 0.891:
    yi_norm = yi_norm * (0.891 / peak_i)

sf.write(str(INST_FINAL), yi_norm, sri, subtype='PCM_24')
print(f"\n✅ Instrumental FINAL: {INST_FINAL.name}")
print(f"   Size: {INST_FINAL.stat().st_size/1e6:.1f} MB")

In [ ]:
# ============================================================
# CELL 12: KIỂM TRA CHẤT LƯỢNG — WAVEFORM + SPECTRUM
# Visual QC để phát hiện artifact trước khi download
# ============================================================
import matplotlib.pyplot as plt
import librosa
import librosa.display
import numpy as np

fig, axes = plt.subplots(3, 2, figsize=(18, 12))
fig.suptitle('QC: Vocal Separation — Waveform & Mel Spectrogram', fontsize=14, fontweight='bold')

files_to_check = [
    (RAW_AUDIO_ACTUAL, "Original (Raw)"),
    (VOCAL_FINAL,      "Vocal FINAL (after all passes + post)"),
    (INST_FINAL,       "Instrumental FINAL"),
]

for i, (fpath, title) in enumerate(files_to_check):
    if not fpath.exists():
        continue

    y, sr = librosa.load(str(fpath), sr=44100, duration=60)  # Lấy 60s đầu để check

    # Waveform
    ax_wave = axes[i][0]
    librosa.display.waveshow(y, sr=sr, ax=ax_wave, alpha=0.7, color='#2196F3')
    ax_wave.set_title(f"{title}\nWaveform", fontsize=10)
    ax_wave.set_xlabel("Time (s)")
    ax_wave.set_ylabel("Amplitude")

    # Mel Spectrogram
    ax_spec = axes[i][1]
    D = librosa.amplitude_to_db(
        np.abs(librosa.stft(y, n_fft=2048, hop_length=512)),
        ref=np.max
    )
    img = librosa.display.specshow(D, sr=sr, hop_length=512,
                                    x_axis='time', y_axis='hz',
                                    ax=ax_spec, cmap='magma')
    ax_spec.set_title(f"{title}\nSpectrogram", fontsize=10)
    ax_spec.set_ylim(0, 8000)  # Tập trung vào range vocal 0–8kHz
    plt.colorbar(img, ax=ax_spec, format='%+2.0f dB')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "QC_spectrogram.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ QC chart đã lưu: QC_spectrogram.png")

In [ ]:
# ============================================================
# CELL 13: NGHE THỬ TRỰC TIẾP TRONG NOTEBOOK
# ============================================================
from IPython.display import Audio, display
import librosa

SR = 44100
PREVIEW_DURATION = 45  # giây

files_preview = [
    (RAW_AUDIO_ACTUAL, "🎧 Original (Raw)"),
    (VOCAL_FINAL,      "🎤 Vocal FINAL"),
    (INST_FINAL,       "🎵 Instrumental FINAL"),
]

for fpath, label in files_preview:
    if fpath.exists():
        print(f"\n{label}")
        y_preview, _ = librosa.load(str(fpath), sr=SR,
                                     duration=PREVIEW_DURATION,
                                     offset=30)  # Bắt đầu từ giây 30 (thường có vocal)
        display(Audio(y_preview, rate=SR))
    else:
        print(f"⚠️ Không tìm thấy: {fpath}")

In [ ]:
# ============================================================
# CELL 14: ĐÓNG GÓI VÀ TẢI VỀ
# Zip toàn bộ output → 1 file duy nhất để tải về
# ============================================================
import zipfile
import os
from pathlib import Path

ZIP_PATH = BASE_DIR / f"{PROJECT_NAME}_UVR5_MDX23C_output.zip"

FILES_TO_PACKAGE = [
    VOCAL_FINAL,
    INST_FINAL,
    OUTPUT_DIR / "QC_spectrogram.png",
]

with zipfile.ZipFile(str(ZIP_PATH), 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in FILES_TO_PACKAGE:
        if f.exists():
            zf.write(str(f), f.name)
            print(f"   + {f.name}")
        else:
            print(f"   ⚠️ Không tìm thấy: {f.name}")

zip_size = ZIP_PATH.stat().st_size / 1e6
print(f"\n✅ Đóng gói xong: {ZIP_PATH.name} ({zip_size:.1f} MB)")
print(f"   Đường dẫn đầy đủ: {ZIP_PATH}")
print("\n→ Vào tab [Output] bên trái Kaggle để tải file về.")

In [ ]:
# ============================================================
# CELL 15: TÓM TẮT KẾT QUẢ
# ============================================================
import soundfile as sf

print("=" * 65)
print("  KẾT QUẢ PIPELINE UVR5 MDX23C")
print("=" * 65)

summary = [
    ("Pass 1", "MDX23C-8KFFT-3_7C",       "Vocal / Instrumental split"),
    ("Pass 2", "UVR-DeEcho-DeReverb",       "Loại echo & reverb"),
    ("Pass 3", "UVR-BVE-4B_SRVgg",          "Tách lead / backing vocal"),
    ("Post",   "DeNoise + HP Filter + LUFS", "Final cleanup & normalize"),
]
for step, model, desc in summary:
    print(f"  [{step}] {model:35s} → {desc}")

print("\nOutput files:")
for fpath in [VOCAL_FINAL, INST_FINAL]:
    if fpath.exists():
        info = sf.info(str(fpath))
        print(f"  ✅ {fpath.name}")
        print(f"      Duration : {info.duration:.1f}s")
        print(f"      SR       : {info.samplerate}Hz")
        print(f"      Format   : {info.format} / {info.subtype}")
        print(f"      Size     : {fpath.stat().st_size/1e6:.1f} MB")

print("\n" + "=" * 65)
print("  📁 Tải về: tab [Output] trong Kaggle sidebar")
print(f"  📦 ZIP    : {ZIP_PATH.name}")
print("=" * 65)

---
## 🛠️ TROUBLESHOOTING — Xử lý lỗi thường gặp

### ❌ `CUDA out of memory`
```python
# Giảm batch_size trong mdx23c_params:
"batch_size": 1,  # từ 2 xuống 1
# Hoặc giảm segment_size:
"segment_size": 128,  # từ 256 xuống 128
```

### ❌ Model không tải được (`Connection timeout`)
```python
# Dùng HuggingFace mirror thay vì GitHub:
# Tải thủ công từ: https://huggingface.co/Anjok07/ultimatevocalremovergui/tree/main/models
# Upload lên Kaggle Dataset → mount vào /kaggle/input/
# Sau đó set: model_file_dir="/kaggle/input/uvr5-models/"
```

### ❌ `audio-separator` không nhận model MDX23C
```bash
# Kiểm tra version:
!pip show audio-separator
# Cần >= 0.21.0 cho MDX23C
!pip install -q "audio-separator[gpu]">=0.24.0
```

### ❌ Vocal còn lẫn nhạc
```python
# Thử thay model Pass 1 bằng model khác (chọn 1 trong 3):
# "MDX23C-8KFFT-3_7C.ckpt"          ← tốt nhất, đang dùng
# "MDX23C_D1064-8KFFT.ckpt"         ← alternative
# "UVR-MDX-NET-Voc_FT.onnx"         ← fallback nếu 2 cái trên fail

# Hoặc tăng aggression trong Pass 2:
"aggression": 15,  # từ 10 lên 15 (loại reverb mạnh hơn)
```

### ❌ Vocal bị méo / artifact
```python
# Giảm aggression trong DeNoise:
prop_decrease=0.3,  # từ 0.55 xuống 0.3 (nhẹ tay hơn)

# Hoặc tắt DeNoise hoàn toàn và chỉ dùng UVR passes
```

---
## 📊 So sánh model MDX23C vs các lựa chọn khác

| Model | Vocal Isolation | Tốc độ GPU | VRAM | Ghi chú |
|-------|----------------|-----------|------|--------|
| **MDX23C-8KFFT-3_7C** | ⭐⭐⭐⭐⭐ | Trung bình | ~4GB | **Tốt nhất — đang dùng** |
| MDX23C_D1064-8KFFT | ⭐⭐⭐⭐ | Nhanh | ~3GB | Backup tốt |
| UVR-MDX-NET-Voc_FT | ⭐⭐⭐⭐ | Nhanh | ~2GB | Dùng khi VRAM giới hạn |
| Demucs htdemucs_ft | ⭐⭐⭐⭐⭐ | Chậm | ~6GB | Chất lượng tương đương, chậm hơn 3x |
| UVR-BVE-4B_SRVgg | ⭐⭐⭐ (backing) | Nhanh | ~2GB | Chỉ dùng cho step tách backing |

---
**Output chuẩn để đưa vào CapCut / DaVinci Resolve:**
- File: `WAV 24-bit / 44.1kHz`  
- Loudness: `-16 LUFS` (chuẩn YouTube Shorts)  
- Peak: `≤ -1 dBFS`